# EDA - Classifying Spam Emails

Notebook nay phan tich du lieu theo dung bai toan spam/not spam. Neu cac file processed lon chua duoc tai qua Git LFS, notebook dung raw SpamAssassin co san trong repo de van chay duoc.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from config import RAW_DATA_DIR, SPAMASSASSIN_ARCHIVES, FIGURES_DIR
from src.data_loader import parse_email_file, iter_email_paths
from src.feature_engineering import extract_manual_features

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_SAMPLE_PATH = FIGURES_DIR / "manual_feature_sample.csv"

def build_spamassassin_feature_sample():
    rows = []
    for source in SPAMASSASSIN_ARCHIVES:
        extract_dir = RAW_DATA_DIR / Path(source["file_name"]).stem.replace(".tar", "")
        if not extract_dir.exists():
            continue
        for email_path in iter_email_paths(extract_dir):
            try:
                _, text = parse_email_file(email_path)
            except Exception:
                continue
            if len(text.strip()) >= 20:
                rows.append({"label": int(source["label"]), "text": text})
    if not rows:
        raise FileNotFoundError("No processed CSV and no readable SpamAssassin raw data.")
    raw_df = pd.DataFrame(rows).drop_duplicates(subset=["label", "text"]).reset_index(drop=True)
    manual_df = extract_manual_features(raw_df["text"].fillna("").astype(str).tolist())
    feature_df = pd.concat([manual_df, raw_df[["label"]]], axis=1)
    feature_df.to_csv(FEATURE_SAMPLE_PATH, index=False, encoding="utf-8")
    return feature_df

if FEATURE_SAMPLE_PATH.exists():
    feature_df = pd.read_csv(FEATURE_SAMPLE_PATH)
else:
    feature_df = build_spamassassin_feature_sample()

print("Emails used for EDA:", len(feature_df))
display(feature_df.head())

## 1. Phan phoi nhan

Kiem tra phan phoi nhan giup biet du lieu co lech lop hay khong. Voi spam detection, du lieu lech lop co the lam Accuracy nhin cao nhung Recall spam thap.

In [ ]:
label_counts = feature_df["label"].value_counts().sort_index().rename(index={0: "not spam", 1: "spam"})
label_percent = (label_counts / label_counts.sum() * 100).round(2)
display(pd.DataFrame({"count": label_counts, "percent": label_percent}))

ax = label_counts.plot(kind="bar", color=["#4C78A8", "#D65F5F"], figsize=(5.8, 4))
ax.set_title("Spam/not spam label distribution")
ax.set_xlabel("Label")
ax.set_ylabel("Email count")
ax.set_xticklabels(label_counts.index, rotation=0)
plt.tight_layout()
plt.show()

## 2. Phan phoi do dai va feature thu cong

Cac feature thu cong bam sat de bai: do dai email, so URL, so dau `!`, so ky tu `$`, va ty le chu hoa.

In [ ]:
display(feature_df.describe().T)

fig, ax = plt.subplots(figsize=(8, 4.5))
for label, name, color in [(0, "not spam", "#4C78A8"), (1, "spam", "#D65F5F")]:
    feature_df.loc[feature_df["label"] == label, "length"].clip(upper=10000).plot(
        kind="hist", bins=50, alpha=0.55, label=name, color=color, ax=ax
    )
ax.set_title("Email length distribution by label")
ax.set_xlabel("Email length (clipped at 10,000 characters)")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Correlation heatmap

Heatmap duoi day cho thay quan he tuyen tinh giua cac feature thu cong va nhan `label`. Day la buoc EDA, khong thay the danh gia model.

In [ ]:
corr = feature_df.corr(numeric_only=True)
corr_path = FIGURES_DIR / "manual_feature_correlation_matrix.csv"
heatmap_path = FIGURES_DIR / "manual_feature_correlation_heatmap.png"
corr.to_csv(corr_path, encoding="utf-8")

fig, ax = plt.subplots(figsize=(7.2, 5.6))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Pearson correlation")
ax.set_xticks(range(len(corr.columns)), labels=corr.columns, rotation=35, ha="right")
ax.set_yticks(range(len(corr.index)), labels=corr.index)
ax.set_title("Correlation heatmap - manual email features")
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        value = corr.iloc[i, j]
        ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(value) > 0.55 else "black")
fig.tight_layout()
fig.savefig(heatmap_path, dpi=180)
plt.show()
print("Saved:", heatmap_path)

## Nhan xet nhanh

Trong mau SpamAssassin raw, `exclamation_count` va `uppercase_ratio` co tuong quan duong ro hon voi nhan spam so voi cac feature con lai. `url_count` gan nhu khong noi bat trong corpus nay, nen khong nen ket luan URL luon vo dung; no co the manh hon o dataset phishing/quang cao hien dai.